### Feature Engineering and Forecast Folds

Build causal log-return features and exact-calendar cumulative log-change targets from the WFP panel. This notebook creates observed-endpoint supervised tables and chronological fold definitions for H1 and H2; it does not fit models, scalers, or encoders.

In [1]:
import base64
import os
from importlib import reload
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd

try:
    import google.colab  # type: ignore
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    from dotenv import load_dotenv
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
    from dotenv import load_dotenv

REPO_URL = 'https://github.com/RAM-AS-RAP/ucs35-mudibo-ml.git'
REPO_BRANCH = 'feature/feature-engineering'

if IN_COLAB:
    drive.mount('/content/drive')
    ENV_PATHS = [
        Path('/content/drive/MyDrive/capstone-data/.env'),
        Path('/content/drive/MyDrive/AgriSence/.env'),
        Path('/content/drive/MyDrive/.env'),
        Path('/content/.env'),
    ]
    for env_path in ENV_PATHS:
        if env_path.exists():
            load_dotenv(env_path, override=False)

    github_token = os.getenv('GITHUB_TOKEN') or os.getenv('GH_TOKEN')
    try:
        from google.colab import userdata
        github_token = github_token or userdata.get('GITHUB_TOKEN') or userdata.get('GH_TOKEN')
    except Exception:
        pass
else:
    ENV_PATHS = []
    github_token = None

repo_dir = Path('/content/AgriSence') if IN_COLAB else None
repo_candidates = [repo_dir, Path.cwd()] if IN_COLAB else [Path.cwd(), *Path.cwd().resolve().parents]
repo_root = next(
    (
        path.resolve() for path in repo_candidates
        if path is not None
        and (path / 'pyproject.toml').exists()
        and (path / 'src' / 'agrisence').exists()
    ),
    None,
)

if repo_root is None and IN_COLAB:
    if not github_token:
        searched_paths = "\n".join(f'- {path}' for path in ENV_PATHS)
        raise RuntimeError(
            'Private repository clone requires GITHUB_TOKEN or GH_TOKEN. '
            'Add it to Colab Secrets (with notebook access enabled), or to a Drive .env file. '
            'Secrets could not be read in this runtime. Checked:\n'
            f'{searched_paths}'
        )
    if repo_dir.exists():
        repo_dir = Path('/content/AgriSence_feature_engineering')
    auth_header = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
    subprocess.run(
        [
            'git', '-c',
            f'http.https://github.com/.extraheader=AUTHORIZATION: basic {auth_header}',
            'clone', '-b', REPO_BRANCH, REPO_URL, str(repo_dir),
        ],
        check=True,
    )
    repo_root = repo_dir.resolve()

if repo_root is None:
    raise FileNotFoundError('Start this notebook from inside the AgriSence repository.')

if IN_COLAB:
    os.chdir(repo_root)

load_dotenv(repo_root / '.env', override=False)
sys.path.insert(0, str(repo_root / 'src'))

from agrisence.preprocessing import feature_engineering
from agrisence.training.folds import build_expanding_window_folds

reload(feature_engineering)

if IN_COLAB:
    panel_path = Path(
        '/content/drive/MyDrive/capstone-data/'
        'cleaned_monthly_prices_wfp_2010_2019-10.csv'
    )
    drive_output_dir = Path('/content/drive/MyDrive/capstone-data/feature_engineering')
else:
    panel_path = repo_root / 'results/artifacts/cleaned_monthly_prices_wfp_2010_2019-10.csv'
    drive_output_dir = None

artifact_dir = repo_root / 'results/artifacts/feature_engineering'
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)
print(f'Repository: {repo_root}')
print(f'Canonical panel: {panel_path}')
print(f'Local artifact directory: {artifact_dir}')

Mounted at /content/drive
Repository: /content/AgriSence
Canonical panel: /content/drive/MyDrive/capstone-data/cleaned_monthly_prices_wfp_2010_2019-10.csv
Local artifact directory: /content/AgriSence/results/artifacts/feature_engineering


## 1. Load the canonical WFP panel

The panel is normalized to month starts and sorted within each market-commodity series.

In [2]:
series_cols = ('market', 'commodity')
required_columns = {
    *series_cols,
    'month',
    'avg_price_per_kg',
    'impute_method',
}

if not panel_path.exists():
    raise FileNotFoundError(f'Canonical WFP panel was not found: {panel_path}')

panel = pd.read_csv(panel_path)
missing_columns = required_columns.difference(panel.columns)
assert not missing_columns, f'Missing canonical panel columns: {sorted(missing_columns)}'
panel['month'] = pd.to_datetime(panel['month'], errors='raise')
panel['month'] = pd.to_datetime({
    'year': panel['month'].dt.year,
    'month': panel['month'].dt.month,
    'day': 1,
})
panel = panel.sort_values([*series_cols, 'month']).reset_index(drop=True)
assert len(panel.groupby(list(series_cols))) == 8
assert not panel.duplicated([*series_cols, 'month']).any()
assert panel['month'].min() < pd.Timestamp('2010-01-01')
assert panel['month'].max() == pd.Timestamp('2019-10-01')

continuity = panel.groupby(list(series_cols))['month'].agg(
    first='min', last='max', count='size'
).reset_index()
for row in continuity.itertuples(index=False):
    expected = pd.date_range(row.first, row.last, freq='MS')
    actual = panel.loc[
        panel['market'].eq(row.market) & panel['commodity'].eq(row.commodity), 'month'
    ].reset_index(drop=True)
    assert actual.equals(pd.Series(expected)), f'Non-contiguous monthly index for {row.market} {row.commodity}'

display(continuity)
print(f'Loaded {len(panel)} rows; pre-2010 warm-up history retained.')

,market,commodity,first,last,count
0,Kisumu,Beans,2006-01-01,2019-10-01,166
1,Kisumu,Maize,2006-01-01,2019-10-01,166
2,Mombasa,Beans,2006-01-01,2019-10-01,166
3,Mombasa,Maize,2006-01-01,2019-10-01,166
4,Nairobi,Beans,2006-01-01,2019-10-01,166
5,Nairobi,Maize,2006-01-01,2019-10-01,166
6,Nakuru,Beans,2008-01-01,2019-10-01,142
7,Nakuru,Maize,2008-02-01,2019-10-01,141


Loaded 1279 rows; pre-2010 warm-up history retained.


### 2.Build causal features on the full panel

Features are computed within each series. Log-return lags use shifted log prices; trailing 3- and 12-month ratios exclude the current month. Calendar and market/commodity indicator columns are appended without fitting encoders.

In [3]:
feature_engineering = reload(feature_engineering)
feature_panel = feature_engineering.build_features(panel, series_cols=series_cols)
display(feature_panel[[
    *series_cols, 'month', 'avg_price_per_kg', 'log_price',
    'log_return_lag_1', 'log_return_lag_2', 'log_return_lag_3',
    'log_ratio_3m', 'log_ratio_12m', 'month_sin', 'month_cos', 'year',
    'market_Nairobi', 'market_Kisumu', 'market_Mombasa', 'market_Nakuru',
    'commodity_Maize', 'commodity_Beans',
]].head(15))

,market,commodity,month,avg_price_per_kg,log_price,log_return_lag_1,log_return_lag_2,log_return_lag_3,log_ratio_3m,log_ratio_12m,month_sin,month_cos,year,market_Nairobi,market_Kisumu,market_Mombasa,market_Nakuru,commodity_Maize,commodity_Beans
0,Kisumu,Beans,2006-01-01,42.027222,3.738318,NaN,NaN,NaN,NaN,NaN,0.000000e+00,1.000000e+00,2006,0,1,0,0,0,1
1,Kisumu,Beans,2006-02-01,43.617222,3.775452,NaN,NaN,NaN,NaN,NaN,5.000000e-01,8.660254e-01,2006,0,1,0,0,0,1
2,Kisumu,Beans,2006-03-01,52.343333,3.957825,0.037135,NaN,NaN,NaN,NaN,8.660254e-01,5.000000e-01,2006,0,1,0,0,0,1
3,Kisumu,Beans,2006-04-01,50.852778,3.928935,0.182373,0.037135,NaN,0.133960,NaN,1.000000e+00,6.123234e-17,2006,0,1,0,0,0,1
4,Kisumu,Beans,2006-05-01,49.283889,3.897597,-0.028890,0.182373,0.037135,0.041531,NaN,8.660254e-01,-5.000000e-01,2006,0,1,0,0,0,1
5,Kisumu,Beans,2006-06-01,37.810556,3.632588,-0.031338,-0.028890,0.182373,-0.030522,NaN,5.000000e-01,-8.660254e-01,2006,0,1,0,0,0,1
6,Kisumu,Beans,2006-07-01,35.162778,3.559988,-0.265009,-0.031338,-0.028890,-0.187118,NaN,1.224647e-16,-1.000000e+00,2006,0,1,0,0,0,1
7,Kisumu,Beans,2006-08-01,30.487778,3.417326,-0.072600,-0.265009,-0.031338,-0.136736,NaN,-5.000000e-01,-8.660254e-01,2006,0,1,0,0,0,1
8,Kisumu,Beans,2006-09-01,30.986667,3.433557,-0.142662,-0.072600,-0.265009,-0.119308,NaN,-8.660254e-01,-5.000000e-01,2006,0,1,0,0,0,1
9,Kisumu,Beans,2006-10-01,31.520000,3.450622,0.016231,-0.142662,-0.072600,-0.036733,NaN,-1.000000e+00,-1.836970e-16,2006,0,1,0,0,0,1


### 3. Build exact-calendar H1 and H2 targets

Targets are cumulative log changes from the origin price to the price exactly one or two calendar months later. Target construction runs on the intact monthly panel, before eligibility or date-window filtering.

In [4]:
feature_engineering = reload(feature_engineering)
target_panel = feature_engineering.build_targets(
    feature_panel,
    series_cols=series_cols,
)
target_columns = [
    *series_cols, 'month', 'avg_price_per_kg', 'target_month_h1',
    'target_h1', 'target_method_h1', 'target_month_h2', 'target_h2',
    'target_method_h2',
]
display(target_panel.loc[:, target_columns].head(15))

,market,commodity,month,avg_price_per_kg,target_month_h1,target_h1,target_method_h1,target_month_h2,target_h2,target_method_h2
0,Kisumu,Beans,2006-01-01,42.027222,2006-02-01,0.037135,observed,2006-03-01,0.219507,observed
1,Kisumu,Beans,2006-02-01,43.617222,2006-03-01,0.182373,observed,2006-04-01,0.153483,observed
2,Kisumu,Beans,2006-03-01,52.343333,2006-04-01,-0.028890,observed,2006-05-01,-0.060227,observed
3,Kisumu,Beans,2006-04-01,50.852778,2006-05-01,-0.031338,observed,2006-06-01,-0.296346,observed
4,Kisumu,Beans,2006-05-01,49.283889,2006-06-01,-0.265009,observed,2006-07-01,-0.337609,observed
5,Kisumu,Beans,2006-06-01,37.810556,2006-07-01,-0.072600,observed,2006-08-01,-0.215262,observed
6,Kisumu,Beans,2006-07-01,35.162778,2006-08-01,-0.142662,observed,2006-09-01,-0.126431,observed
7,Kisumu,Beans,2006-08-01,30.487778,2006-09-01,0.016231,observed,2006-10-01,0.033296,observed
8,Kisumu,Beans,2006-09-01,30.986667,2006-10-01,0.017065,observed,2006-11-01,0.078370,observed
9,Kisumu,Beans,2006-10-01,31.520000,2006-11-01,0.061305,observed,2006-12-01,0.149866,observed


### 4. Guard features and targets against calendar gaps

The guard masks lagged returns, trailing ratios, and horizon targets when a required source month is absent. Imputed observations may remain in feature history; eligibility filtering below separately requires observed origin and target prices.

In [5]:
feature_engineering = reload(feature_engineering)
guarded_panel = feature_engineering.apply_calendar_distance_guard(
    target_panel,
    series_cols=series_cols,
)
print('Rows failing at least one history-distance check:', int((~guarded_panel['consecutive_observation']).sum()))
print('Null targets after exact-distance guard:', {
    'h1': int(guarded_panel['target_h1'].isna().sum()),
    'h2': int(guarded_panel['target_h2'].isna().sum()),
})
display(guarded_panel[[
    *series_cols, 'month', 'consecutive_observation',
    'log_return_lag_1', 'log_ratio_3m', 'log_ratio_12m',
    'target_h1', 'target_h2',
]].head(15))

Rows failing at least one history-distance check: 96
Null targets after exact-distance guard: {'h1': 8, 'h2': 16}


,market,commodity,month,consecutive_observation,log_return_lag_1,log_ratio_3m,log_ratio_12m,target_h1,target_h2
0,Kisumu,Beans,2006-01-01,False,NaN,NaN,NaN,0.037135,0.219507
1,Kisumu,Beans,2006-02-01,False,NaN,NaN,NaN,0.182373,0.153483
2,Kisumu,Beans,2006-03-01,False,0.037135,NaN,NaN,-0.028890,-0.060227
3,Kisumu,Beans,2006-04-01,False,0.182373,0.133960,NaN,-0.031338,-0.296346
4,Kisumu,Beans,2006-05-01,False,-0.028890,0.041531,NaN,-0.265009,-0.337609
5,Kisumu,Beans,2006-06-01,False,-0.031338,-0.030522,NaN,-0.072600,-0.215262
6,Kisumu,Beans,2006-07-01,False,-0.265009,-0.187118,NaN,-0.142662,-0.126431
7,Kisumu,Beans,2006-08-01,False,-0.072600,-0.136736,NaN,0.016231,0.033296
8,Kisumu,Beans,2006-09-01,False,-0.142662,-0.119308,NaN,0.017065,0.078370
9,Kisumu,Beans,2006-10-01,False,0.016231,-0.036733,NaN,0.061305,0.149866


### 5. Build observed-eligible supervised tables

Construct separate direct H1 and H2 tables with origins from January 2010 through October 2019. Primary rows require `impute_method == 'observed'` at both origin and target. The functions report endpoint exclusions separately from missing-feature exclusions.

In [6]:
feature_engineering = reload(feature_engineering)
model_df_h1 = feature_engineering.build_supervised_table(
    guarded_panel,
    'h1',
    window_start='2010-01-01',
    window_end='2019-10-01',
)
model_df_h2 = feature_engineering.build_supervised_table(
    guarded_panel,
    'h2',
    window_start='2010-01-01',
    window_end='2019-10-01',
)
print('H1 supervised rows:', len(model_df_h1))
print('H2 supervised rows:', len(model_df_h2))

h1: rows excluded for non-observed/missing endpoints: 31
h1: origins in window=944; excluded for missing features=0; final supervised rows=913
h2: rows excluded for non-observed/missing endpoints: 40
h2: origins in window=944; excluded for missing features=0; final supervised rows=904
H1 supervised rows: 913
H2 supervised rows: 904


### 6. Build expanding-window folds

Fold masks assign origin months to development or final test and are shared across all series. H1/H2 training-target cutoffs are embargoed by each forecast horizon so labels are available at the earliest validation origin.

In [7]:
folds_h1 = build_expanding_window_folds(model_df_h1)
folds_h2 = build_expanding_window_folds(model_df_h2)

fold_summary = pd.DataFrame([
    {
        'horizon': horizon,
        'supervised_rows': len(model_df),
        'development_rows': int(folds['development_mask'].sum()),
        'test_rows': int(folds['test_mask'].sum()),
        'outside_dev_test_rows': int((~folds['development_mask'] & ~folds['test_mask']).sum()),
        'expanding_folds': len(folds['folds']),
    }
    for horizon, model_df, folds in (
        ('h1', model_df_h1, folds_h1),
        ('h2', model_df_h2, folds_h2),
    )
])
display(fold_summary)
print('H1 fold boundaries:')
display(pd.DataFrame(folds_h1['folds']))
print('H2 fold boundaries:')
display(pd.DataFrame(folds_h2['folds']))

,horizon,supervised_rows,development_rows,test_rows,outside_dev_test_rows,expanding_folds
0,h1,913,825,88,0,4
1,h2,904,824,80,0,4


H1 fold boundaries:


,train_origin_start,train_origin_end,train_target_end,validation_start,validation_end
0,2010-01-01,2014-12-01,2015-01-01,2015-01-01,2015-12-01
1,2010-01-01,2015-12-01,2016-01-01,2016-01-01,2016-12-01
2,2010-01-01,2016-12-01,2017-01-01,2017-01-01,2017-12-01
3,2010-01-01,2017-12-01,2018-01-01,2018-01-01,2018-10-01


H2 fold boundaries:


,train_origin_start,train_origin_end,train_target_end,validation_start,validation_end
0,2010-01-01,2014-11-01,2015-01-01,2015-01-01,2015-12-01
1,2010-01-01,2015-11-01,2016-01-01,2016-01-01,2016-12-01
2,2010-01-01,2016-11-01,2017-01-01,2017-01-01,2017-12-01
3,2010-01-01,2017-11-01,2018-01-01,2018-01-01,2018-10-01


### 7. Summarize and save artifacts

Save each supervised table as CSV and the corresponding row masks and expanding-fold boundaries as JSON. In Colab, write to Drive as well as the repository artifact folder for persistence. This stage only constructs data products; it does not train or fit any estimator or transform.

In [8]:
import json

artifact_dir.mkdir(parents=True, exist_ok=True)
if IN_COLAB:
    drive_output_dir.mkdir(parents=True, exist_ok=True)


def _fold_payload(model_df, fold_result):
    return {
        'table_row_position': list(range(len(model_df))),
        'source_panel_index': [
            int(value) if isinstance(value, (int, np.integer)) else str(value)
            for value in model_df.index
        ],
        'development_mask': fold_result['development_mask'].astype(bool).tolist(),
        'test_mask': fold_result['test_mask'].astype(bool).tolist(),
        'folds': [
            {key: value.strftime('%Y-%m-%d') for key, value in fold.items()}
            for fold in fold_result['folds']
        ],
    }


saved_paths = []
for horizon, model_df, fold_result in (
    ('h1', model_df_h1, folds_h1),
    ('h2', model_df_h2, folds_h2),
):
    table = model_df.copy()
    development_mask = fold_result['development_mask']
    test_mask = fold_result['test_mask']
    table['fold_split'] = np.select(
        [development_mask.to_numpy(), test_mask.to_numpy()],
        ['development', 'test'],
        default='outside_dev_test_window',
    )
    table_path = artifact_dir / f'model_df_{horizon}.csv'
    folds_path = artifact_dir / f'folds_{horizon}.json'
    table.to_csv(table_path, index=False)
    folds_path.write_text(json.dumps(_fold_payload(model_df, fold_result), indent=2))
    saved_paths.extend([table_path, folds_path])

    if IN_COLAB:
        drive_table_path = drive_output_dir / table_path.name
        drive_folds_path = drive_output_dir / folds_path.name
        table.to_csv(drive_table_path, index=False)
        drive_folds_path.write_text(json.dumps(_fold_payload(model_df, fold_result), indent=2))
        saved_paths.extend([drive_table_path, drive_folds_path])

summary_path = artifact_dir / 'feature_engineering_split_summary.csv'
fold_summary.to_csv(summary_path, index=False)
saved_paths.append(summary_path)
if IN_COLAB:
    drive_summary_path = drive_output_dir / summary_path.name
    fold_summary.to_csv(drive_summary_path, index=False)
    saved_paths.append(drive_summary_path)

for path in saved_paths:
    print(f'{path}: exists={path.exists()}, bytes={path.stat().st_size}')

assert all(path.exists() and path.stat().st_size > 0 for path in saved_paths)


/content/AgriSence/results/artifacts/feature_engineering/model_df_h1.csv: exists=True, bytes=344011
/content/AgriSence/results/artifacts/feature_engineering/folds_h1.json: exists=True, bytes=36638
/content/drive/MyDrive/capstone-data/feature_engineering/model_df_h1.csv: exists=True, bytes=344011
/content/drive/MyDrive/capstone-data/feature_engineering/folds_h1.json: exists=True, bytes=36638
/content/AgriSence/results/artifacts/feature_engineering/model_df_h2.csv: exists=True, bytes=340730
/content/AgriSence/results/artifacts/feature_engineering/folds_h2.json: exists=True, bytes=36284
/content/drive/MyDrive/capstone-data/feature_engineering/model_df_h2.csv: exists=True, bytes=340730
/content/drive/MyDrive/capstone-data/feature_engineering/folds_h2.json: exists=True, bytes=36284
/content/AgriSence/results/artifacts/feature_engineering/feature_engineering_split_summary.csv: exists=True, bytes=125
/content/drive/MyDrive/capstone-data/feature_engineering/feature_engineering_split_summary.cs